# step A-1 — scaleup-500 (RQ1 대조군, distinct 선행, 500 확대 재실행)

**대응 RQ:** RQ1 대조 — stepA(clone)의 절벽·자기증폭이 선행을 **서로 다른 과제 12개(distinct)** 로 바꿔도 재현되는가.

**무엇을 확인하나**
- stepA와 **composition만 다름**(clone→distinct). 나머지(camel·약함·8~0·seed56·생성3) 동일.
- distinct에서도 같은 절벽·자기증폭이면 → '데모 오버라이드'가 아니라 **형태 신호** → RQ1 견고.

**파일럿 대비:** 준수 격자 4~0→**8~0**, seed 20→**56**, 총 100→**≈500**. 확대 재실행이다.

**코드 동일 보장(재현성):** 실험 로직을 쓰지 않고 파일럿과 **똑같은 `harness.run`을 호출**만 한다.
겹치는 부분(**준수 4~0 × seed 0~19, distinct**)은 파일럿(`results/stepA-1/`)과 **동일**하게 나온다.

설계 문서: `docs/stepA-1/scaleup-500.md` (파일럿: `docs/stepA-1/plan.md`).

> **메모리·시간(무료 T4):** Qwen2.5-Coder-3B-Instruct fp16 ≈ 6.2GB로 T4(15GB)에 적재 가능.
> **504회 × 3턴** 생성이라 파일럿(100회)의 약 5배 — 대략 **1.5~3.5시간**. 양자화 미사용(fp16).
> **재개 가능:** 조건마다 즉시 저장하고 이미 저장된 조건은 건너뛴다. 셀 3·4·5 재실행하면 남은 조건부터 이어서 한다.
> **clone 비교:** 요약(셀 7)은 `results/stepA_scaleup500/`(stepA scaleup)와 겹쳐 그린다. 먼저 stepA scaleup을 돌려두면 대비가 보인다.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepA-1/scaleup-500
!git checkout stepA-1/scaleup-500
!git pull --quiet origin stepA-1/scaleup-500
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — stepA와 composition만 다르다 (DISTINCT). 값만 확장.
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
N_COMPLIANT = [8, 7, 6, 5, 4, 3, 2, 1, 0]   # 선행 12개 중 camel 준수 개수 (9단계)
SEEDS = list(range(56))                       # 조건당 반복 → 9 x 56 = 504
STEP = 'stepA-1_scaleup500'

def make(n, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=n, composition=Composition.DISTINCT),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        seed=s,
    )

conditions = [make(n, s) for n in N_COMPLIANT for s in SEEDS]

# 실행 전 예측 (결과와 함께 보존, CLAUDE.md §6)
PREDICTION = ('선행이 서로 다른 과제(distinct)여도 clone과 유사한 절벽·자기증폭이면 데모 오버라이드가 아님(형태 신호). '
              '겹치는 조건(4~0 x seed 0~19)은 파일럿과 동일.')
print(len(conditions), '개 조건 =', len(N_COMPLIANT), 'x', len(SEEDS))

In [ ]:
# 실행 — 조건별 순차 생성 + 즉시 저장(재개 가능) + 중간 결과 실시간 표시.
# 실험 로직은 harness.run 이 수행한다(파일럿과 동일 코드). 여기서는 호출·저장·표시만.
import pandas as pd
from IPython.display import clear_output
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

handle = load_model(MODEL)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())

def _row(cond, metrics):
    return {'n_compliant': cond.preceding.n_compliant,
            'compliant': metrics.extra['first_compliant'],
            'first_violated': metrics.extra['first_violated'],
            'subseq_viol': metrics.extra['subsequent_violation_rate']}

def _show(live, done, total, new, skipped):
    df = pd.DataFrame(live)
    rate = df.groupby('n_compliant')['compliant'].mean().reindex(N_COMPLIANT)
    cnt = df.groupby('n_compliant')['compliant'].count().reindex(N_COMPLIANT).fillna(0).astype(int)
    tbl = pd.DataFrame({'준수율': rate.round(3), 'n': cnt})
    clear_output(wait=True)
    print(f'진행 {done}/{total}  (새로 {new} / 건너뜀 {skipped})')
    print('n_compliant별 준수율(첫 함수) — distinct, 지금까지:')
    print(tbl.to_string())

live, new, skipped = [], 0, 0
for i, c in enumerate(conditions):
    p = result_path(c, step=STEP)
    if p.exists():                                   # 이미 한 조건 → 건너뜀(재개)
        live.append(_row(c, load_result(p).metrics)); skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ1', prediction=PREDICTION))
        live.append(_row(out.condition, out.metrics)); new += 1
    if (i + 1) % 14 == 0 or (i + 1) == len(conditions):   # 14 = seed 묶음마다 갱신
        _show(live, i + 1, len(conditions), new, skipped)
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepA-1_scaleup500/ 에 불변 저장된 이 실험 조건들을 모은다
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '건 → results/'+STEP+'/')

In [ ]:
# 요약 — clone(stepA) vs distinct(stepA-1) 준수율 겹쳐 그리기 + 자기증폭
import pandas as pd, matplotlib.pyplot as plt, glob, json

def rate_by_n(step):
    rows = []
    for f in glob.glob(f'results/{step}/*.json'):
        d = json.load(open(f)); e = d['metrics']['extra']
        rows.append({'n': d['condition']['preceding']['n_compliant'], 'c': e['first_compliant']})
    if not rows: return None
    return pd.DataFrame(rows).groupby('n')['c'].mean().reindex(N_COMPLIANT)

d1 = rate_by_n('stepA-1_scaleup500')      # distinct (이 실험)
d0 = rate_by_n('stepA_scaleup500')        # clone (stepA scaleup, 있으면 겹쳐 그림)
print('distinct(A-1) 준수율:'); print(d1.round(3))

# 자기증폭
amp_rows = [{'n': r.condition.preceding.n_compliant,
             'fv': r.metrics.extra['first_violated'],
             'sv': r.metrics.extra['subsequent_violation_rate']} for r in records]
adf = pd.DataFrame(amp_rows)
amp = adf[adf.fv].groupby('n')['sv'].mean()
print('\n자기증폭(첫 위반 조건에서 이후 위반 비율):'); print(amp.round(3))

plt.figure(figsize=(6,3))
plt.plot([str(n) for n in N_COMPLIANT], d1.values, marker='o', label='distinct (A-1)')
if d0 is not None:
    plt.plot([str(n) for n in N_COMPLIANT], d0.values, marker='s', label='clone (A)')
plt.xlabel('n_compliant (8->0)'); plt.ylabel('준수율(첫 함수)'); plt.ylim(-0.02, 1.02)
plt.legend(); plt.grid(alpha=.3); plt.title('RQ1: clone vs distinct (scaleup-500)'); plt.show()